In [2]:
%pip install transformers datasets accelerate evaluate

Note: you may need to restart the kernel to use updated packages.


In [3]:
# IMPORT LIBRARIES

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report
)

import torch
import torch.nn as nn

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

print("Libraries loaded successfully.")

c:\Users\User\anaconda3\envs\mbg\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully.


In [4]:
# LOAD CLEANED DATA

TRAIN_PATH = "../data/interim/train_clean_v2.csv"

train_df = pd.read_csv(TRAIN_PATH)

print(train_df.shape)

display(train_df.head())

(5000, 2)


,full_text,label
0,pret. di sekolah saya dapat mbg tetap saja ...,Sasaran Penerima
1,mbg bentuk penggarongan duit negara secara tsm...,Politik
2,pasal 34 ayat (1) undang-undang dasar negara r...,Sasaran Penerima
3,makan bergizi gratis bikin masyarakat merasa ...,Sasaran Penerima
4,"presiden ngotot, paling kesal kalau mbg dikri...",Politik


In [5]:
# LABEL ENCODING

labels = sorted(train_df['label'].unique())

label2id = {
    label: idx
    for idx, label in enumerate(labels)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

train_df['label_id'] = train_df['label'].map(label2id)

print(label2id)

{'Anggaran': 0, 'Distribusi': 1, 'Ekonomi': 2, 'Kualitas Pangan': 3, 'Lainnya': 4, 'Politik': 5, 'Sasaran Penerima': 6, 'Tata Kelola': 7}


In [6]:
# LOAD TOKENIZER

MODEL_NAME = "indolem/indobertweet-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print(f"Model   : {MODEL_NAME}")
print("Tokenizer loaded.")

Model   : indolem/indobertweet-base-uncased
Tokenizer loaded.


In [7]:
# TOKENIZATION FUNCTION

MAX_LENGTH = 256


def tokenize_function(texts):

    return tokenizer(

        texts.tolist(),

        padding='max_length',

        truncation=True,

        max_length=MAX_LENGTH
    )

In [8]:
# STRATIFIED KFOLD

skf = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42
)

print("StratifiedKFold initialized.")

StratifiedKFold initialized.


In [9]:
# CUSTOM DATASET

from torch.utils.data import Dataset


class TextDataset(Dataset):

    def __init__(
        self,
        encodings,
        labels
    ):

        self.encodings = encodings
        self.labels    = labels


    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item['labels'] = torch.tensor(
            self.labels[idx]
        )

        return item


    def __len__(self):

        return len(self.labels)

In [10]:
# METRIC FUNCTION

import evaluate

accuracy_metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    balanced_acc = balanced_accuracy_score(
        labels,
        predictions
    )

    return {
        "balanced_accuracy": balanced_acc
    }

In [11]:
# WEIGHTED TRAINER

# Hitung class weight dari training data
class_weights_array = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.array(sorted(label2id.values())),
    y            = train_df['label_id'].values
)

CLASS_WEIGHTS = torch.tensor(
    class_weights_array,
    dtype=torch.float
)

print("Class weights:")
for label, idx in sorted(label2id.items(), key=lambda x: x[1]):
    print(f"  {label:20s}: {CLASS_WEIGHTS[idx]:.4f}")


class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs
    ):

        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        loss_fn = nn.CrossEntropyLoss(
            weight=CLASS_WEIGHTS.to(logits.device)
        )

        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

Class weights:
  Anggaran            : 0.8597
  Distribusi          : 1.4434
  Ekonomi             : 4.3103
  Kualitas Pangan     : 0.5012
  Lainnya             : 0.9796
  Politik             : 0.7891
  Sasaran Penerima    : 1.2327
  Tata Kelola         : 1.2231


In [15]:
# CROSS VALIDATION TRAINING

fold_scores     = []
oof_predictions = np.zeros(len(train_df))

for fold, (train_idx, valid_idx) in enumerate(
    skf.split(
        train_df['full_text'],
        train_df['label_id']
    )
):

    print("=" * 60)
    print(f"FOLD {fold + 1}")
    print("=" * 60)

    # SPLIT DATA
    train_fold = train_df.iloc[train_idx]
    valid_fold = train_df.iloc[valid_idx]

    # TOKENIZATION
    train_encodings = tokenize_function(
        train_fold['full_text']
    )

    valid_encodings = tokenize_function(
        valid_fold['full_text']
    )

    # DATASET
    train_dataset = TextDataset(
        train_encodings,
        train_fold['label_id'].tolist()
    )

    valid_dataset = TextDataset(
        valid_encodings,
        valid_fold['label_id'].tolist()
    )

    # MODEL
    model = AutoModelForSequenceClassification.from_pretrained(

        MODEL_NAME,

        num_labels = len(label2id),

        id2label   = id2label,

        label2id   = label2id
    )

    # TRAINING ARGUMENTS
    training_args = TrainingArguments(

        output_dir = f"../models/indobertweet_fold_{fold+1}",

        eval_strategy   = "epoch",
        save_strategy   = "epoch",
        logging_strategy = "epoch",

        num_train_epochs           = 2,

        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 16,

        learning_rate  = 2e-5,
        weight_decay   = 0.01,

        warmup_ratio   = 0.1,

        load_best_model_at_end = True,

        metric_for_best_model = "balanced_accuracy",

        greater_is_better = True,

        report_to = "none"
    )

    # TRAINER — pakai WeightedTrainer
    trainer = WeightedTrainer(

        model = model,

        args  = training_args,

        train_dataset = train_dataset,

        eval_dataset  = valid_dataset,

        compute_metrics = compute_metrics
    )

    # TRAIN
    trainer.train()

    # EVALUATION
    predictions_output = trainer.predict(
        valid_dataset
    )

    predictions = np.argmax(
        predictions_output.predictions,
        axis=-1
    )

    oof_predictions[valid_idx] = predictions

    score = balanced_accuracy_score(
        valid_fold['label_id'],
        predictions
    )

    fold_scores.append(score)

    print(f"Balanced Accuracy : {score:.4f}")
    print()

FOLD 1


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indolem/indobertweet-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [14]:
# OVERALL RESULT

print("=" * 60)
print("INDOBERTWEET RESULT")
print("=" * 60)

for i, score in enumerate(fold_scores):
    print(f"Fold {i+1}: {score:.4f}")

print()
print(f"Mean Balanced Accuracy: {np.mean(fold_scores):.4f}")
print(f"Std Balanced Accuracy : {np.std(fold_scores):.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(
    train_df['label_id'],
    oof_predictions.astype(int),
    target_names=list(label2id.keys())
))

INDOBERTWEET RESULT
Fold 1: 0.6800
Fold 2: 0.6813
Fold 3: 0.7027
Fold 4: 0.7016
Fold 5: 0.6438

Mean Balanced Accuracy: 0.6819
Std Balanced Accuracy : 0.0214

Classification Report:
                  precision    recall  f1-score   support

        Anggaran       0.75      0.82      0.78       727
      Distribusi       0.64      0.67      0.65       433
         Ekonomi       0.65      0.83      0.73       145
 Kualitas Pangan       0.79      0.67      0.72      1247
         Lainnya       0.59      0.60      0.60       638
         Politik       0.64      0.59      0.61       792
Sasaran Penerima       0.64      0.64      0.64       507
     Tata Kelola       0.52      0.65      0.58       511

        accuracy                           0.67      5000
       macro avg       0.65      0.68      0.66      5000
    weighted avg       0.67      0.67      0.67      5000

